In [1]:
# this notebook will serve as a space for developing functions for x-fuse
# it will be used to test out code snippets and ideas before they are added to the main codebase

In [2]:
# Plan out the steps for the x-fuse pipeline below

# STEP 1: GET DINO FEATURES

# STEP 2: UPSAMPLE DINO FEATURES INTEGRATING WITH XRD DATA

# STEP 3: SEGMENT USING UPSAMPLED DINO FEATURES

In [24]:
# imports

import torch

from hr_dv2.utils import *
from hr_dv2 import tr
from hr_dv2 import HighResDV2
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

from typing import Tuple, List, Any, TypedDict, TypeAlias

from hr_dv2.segment import get_dv2_features

from PIL import Image

import imageio.v3 as iio

# 1. Get DINO Features

In [25]:
IMG_SIZE = 518
PATCH_SIZE = 14 #DINOv2
PATCH_H, PATCH_W = IMG_SIZE // PATCH_SIZE, IMG_SIZE // PATCH_SIZE
FEAT_DIM = 384 #DINOv2

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

transform = tr.get_input_transform(IMG_SIZE, IMG_SIZE) #ImageNet normalization

In [26]:
# intialise the HighResDV2 model
dino_model_name = "dinov2_vits14_reg"
net = HighResDV2(dino_model_name, 4, pca_dim=-1, dtype=torch.float16)
net.to(device="mps")

FEAT_DIM = net.get_model_params(dino_model_name)[0]
print(FEAT_DIM)

Using cache found in /Users/trb56939/.cache/torch/hub/facebookresearch_dinov2_main


384


In [ ]:
# load the image

img_path = "../x-fuse/datasets/Zn13X/low_4.tiff"

img = iio.imread(img_path)
img = Image.fromarray(img)


img_cc = tr.closest_crop(img.height, img.width, PATCH_SIZE)

img_tr = tr.get_input_transform(int(640), int(640))

img_tensor, img = tr.load_image(img_path, img_tr)
img_arr = np.array(img)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img_arr)
ax[1].imshow(img_tensor.permute(1, 2, 0).cpu().numpy())
plt.show()

img_tensor =img_tensor.to(torch.float16).to(DEVICE)


AttributeError: module 'PIL.Image' has no attribute 'show'

In [ ]:
shifts = [i for i in range(1,3)]

In [ ]:
# different mechanism for integrating XRD

FUNCTION direct_correlation_gating(features, xrd_map):
    // features: (27, 27, 384)
    // xrd_map: (27, 27, 1)
    
    FOR each channel ch in 0..384:
        correlation[ch] = compute_correlation(
            features[:, :, ch],  // all spatial locations for channel ch
            xrd_map              // XRD intensity map
        )
    END FOR
    
    gates = clamp(correlation, 0, 1)  // (384,)
    
    FOR each spatial location (i, j):
        features[i, j, :] = features[i, j, :] * gates
    END FOR
    
    RETURN features


In [ ]:
CLASS LearnedChannelGating(feat_dim=384):
    
    INIT:
        mlp = neural_network([
            Linear(1 → 128),
            ReLU(),
            Linear(128 → feat_dim),
            Sigmoid()  // output [0, 1]
        ])
    
    FORWARD(features, xrd_map):
        // features: (27, 27, 384)
        // xrd_map: (27, 27, 1)
                FOR each spatial location (i, j):
            xrd_intensity = xrd_map[i, j]  // scalar
            
            gates = mlp(xrd_intensity)  // (384,)
            
            features[i, j, :] = features[i, j, :] * gates
        END FOR
        
        RETURN features
    
    TRAIN:
        FOR each image in dataset:
            gated_features = forward(features, xrd_map)
            upsampled = upsample(gated_features)
            
            loss = correlation_distance(
                upsampled_pca,
                xrd_map
            )
            
            backprop(loss)

In [ ]:
CLASS XRDCrossAttention(feat_dim=384, num_heads=8):
    
    INIT:
        xrd_to_query = Linear(1 → feat_dim)
        attention_layer = MultiHeadAttention(
            embed_dim=feat_dim,
            num_heads=num_heads
        )
    
    FORWARD(features, xrd_map):
        // features: (729, 384)  [flattened 27×27]
        // xrd_map: (729, 1)      [flattened]
        
        // Project XRD to query space
        queries = xrd_to_query(xrd_map)  // (729, 384)
        
        // Attention: "Where does XRD signal attend in feature space?"
        attended_features, attention_weights = attention(
            query = queries,        // What XRD is looking for
            key = features,         // Which features to attend to
            value = features        // What to retrieve
        )  // (729, 384)
        
        RETURN attended_features